# SVGier - Qwen3.5-0.8B LoRA Training (Colab)

This notebook runs the project pipeline end-to-end:
1. Clone repo
2. Install dependencies
3. Build bootstrap dataset (auto-download raw if missing)
4. Train LoRA model manually via `scripts/train.sh`

All code/comments are in English, aligned with project rules.

## 0) Runtime Recommendations
- Runtime type: GPU
- If available, choose an NVIDIA T4/L4/A100 runtime
- For gated datasets/models, set `HF_TOKEN`

In [ ]:
import os
from pathlib import Path

PROJECT_DIR = Path('/content/svgier')
REPO_URL = 'https://github.com/JulioGuillermo/svgier.git'
BRANCH = 'main'

print('PROJECT_DIR =', PROJECT_DIR)

In [ ]:
# Optional: set HF token for better limits and gated assets
HF_TOKEN = os.getenv('HF_TOKEN', '')  # or paste: 'hf_xxx'
if HF_TOKEN:
    os.environ['HF_TOKEN'] = HF_TOKEN
    print('HF_TOKEN is set')
else:
    print('HF_TOKEN is not set (public datasets still work)')

In [ ]:
# Clone repository
!rm -rf /content/svgier
!git clone -b {BRANCH} {REPO_URL} /content/svgier
%cd /content/svgier
!git rev-parse --short HEAD

In [ ]:
# Install uv and dependencies
!pip -q install uv
!uv venv
!uv pip install -r requirements.txt

In [ ]:
# Show detected accelerator from PyTorch
!uv run python - << 'PY'
import torch
dev = 'cpu'
if torch.cuda.is_available():
    dev = 'cuda'
elif getattr(torch, 'xpu', None) is not None and torch.xpu.is_available():
    dev = 'xpu'
elif torch.backends.mps.is_available():
    dev = 'mps'
print('detected_device=', dev)
PY

## 1) Build Bootstrap Dataset
This command auto-downloads raw data if `data/raw/bootstrap_input.jsonl` is missing.

In [ ]:
# Optional limits for faster experiments
TEXT2SVG_LIMIT = 2000
INSTRUCT_SVG_LIMIT = 1000
SVG_EMOJI_LIMIT = 1000

!TEXT2SVG_LIMIT={TEXT2SVG_LIMIT} INSTRUCT_SVG_LIMIT={INSTRUCT_SVG_LIMIT} SVG_EMOJI_LIMIT={SVG_EMOJI_LIMIT} ./scripts/bootstrap_dataset.sh
!ls -lh data/processed
!sed -n '1,80p' data/metadata/bootstrap_data_report.md

## 2) Train LoRA (Manual Script)
You can tune these values depending on your Colab GPU.

In [ ]:
# Training parameters
MODEL_NAME = 'Qwen/Qwen3.5-0.8B'
EPOCHS = 1
MAX_LENGTH = 512
TRAIN_BATCH_SIZE = 1
EVAL_BATCH_SIZE = 1
GRAD_ACCUM = 4
PRECISION = 'bf16'  # use 'fp16' on many CUDA GPUs if needed
OUTPUT_DIR = 'checkpoints/qwen35_colab_run'

!MODEL_NAME={MODEL_NAME} EPOCHS={EPOCHS} MAX_LENGTH={MAX_LENGTH} TRAIN_BATCH_SIZE={TRAIN_BATCH_SIZE} EVAL_BATCH_SIZE={EVAL_BATCH_SIZE} GRAD_ACCUM={GRAD_ACCUM} PRECISION={PRECISION} OUTPUT_DIR={OUTPUT_DIR} ./scripts/train.sh

## 3) Optional: Save Checkpoints to Google Drive

In [ ]:
# Optional checkpoint backup to Drive
from google.colab import drive
drive.mount('/content/drive')

DRIVE_OUT = '/content/drive/MyDrive/svgier_checkpoints'
!mkdir -p {DRIVE_OUT}
!cp -r checkpoints/qwen35_colab_run {DRIVE_OUT}/

## 4) Quick Inference Smoke Test (Optional)

In [ ]:
# Basic check that adapter output exists
!find checkpoints/qwen35_colab_run -maxdepth 2 -type f | head -n 20